<a href="https://colab.research.google.com/github/EmilyHong77/degentrificAItion/blob/main/Final_data_cleaning_geojson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

losing my mind

In [21]:
import geopandas as gpd
from shapely.geometry import MultiPolygon, Polygon

# Path to the .gdb file
gdb_path = '/content/lct_000b21f_e/lct_000b21f_e.gdb'

# Load the desired layer (you found it from the previous step)
layer_name = 'lct_000b21f_e'  # Use the actual layer name you found
gdf = gpd.read_file(gdb_path, layer=layer_name)

# Function to extract coordinates
def extract_coordinates(geometry):
    if isinstance(geometry, Polygon):
        return list(geometry.exterior.coords)
    elif isinstance(geometry, MultiPolygon):
        coords = []
        for poly in geometry.geoms:
            coords.extend(list(poly.exterior.coords))
        return coords
    return None

# Extract coordinates for all polygons
gdf['coords'] = gdf['geometry'].apply(extract_coordinates)

# Filter to get only Montreal CMA CTUIDs
montreal_cma_gdf = gdf[gdf['CTUID'].astype(str).str.startswith('462')]

# Convert the coordinates to lat/long if necessary (depending on CRS)
if montreal_cma_gdf.crs != 'EPSG:4326':
    montreal_cma_gdf = montreal_cma_gdf.to_crs(epsg=4326)

# Display the number of valid geometries
valid_geometries_count = montreal_cma_gdf.shape[0]
print(f"Number of valid geometries in Montreal CMA: {valid_geometries_count}")

# Display the filtered data
print(montreal_cma_gdf[['CTUID', 'coords']].head())


Number of valid geometries in Montreal CMA: 1004
          CTUID                                             coords
161  4620725.07  [(7601049.8829, 1245504.1000000015), (7601042....
162  4620652.04  [(7613072.742899999, 1242515.568599999), (7613...
197  4620887.05  [(7639260.571400002, 1255028.728599999), (7639...
203  4620887.06  [(7639982.577100001, 1255031.634300001), (7639...
220  4620887.07  [(7641359.077100001, 1255838.4028999992), (764...


In [22]:

from pyproj import Transformer

# Define the transformer to convert coordinates to lat/lon (EPSG:4326)
transformer = Transformer.from_crs(gdf.crs, 'EPSG:4326', always_xy=True)

# Function to transform coordinates and format them as lists of lists
def transform_coordinates(geometry):
    if geometry is None:
        return None
    if geometry.geom_type == 'Polygon':
        coords = list(geometry.exterior.coords)
        transformed_coords = [[x, y] for x, y in transformer.itransform(coords)]
        return transformed_coords
    elif geometry.geom_type == 'MultiPolygon':
        transformed_coords = []
        for poly in geometry.geoms:
            coords = list(poly.exterior.coords)
            transformed_coords.append([[x, y] for x, y in transformer.itransform(coords)])
        return transformed_coords
    else:
        return None

# Apply the transformation to all geometries
gdf['latlon_coordinates'] = gdf['geometry'].apply(transform_coordinates)

# Filter to get only Montreal CMA CTUIDs
montreal_cma_gdf = gdf[gdf['CTUID'].astype(str).str.startswith('462')]

# Convert to DataFrame and save as CSV
montreal_cma_df = pd.DataFrame(montreal_cma_gdf[['CTUID', 'latlon_coordinates']])
csv_path = '/content/drive/MyDrive/Ai4Good Project/GentrificAItion/FrontEnd/Final_data/ALL_ctuid_to_latlong.csv'
montreal_cma_df.to_csv(csv_path, index=False)


print(len(montreal_cma_df))
print(montreal_cma_df.head())

1004
          CTUID                                 latlon_coordinates
161  4620725.07  [[[-73.9360593674554, 45.58568488364327], [-73...
162  4620652.04  [[[-73.80088052135274, 45.53047772747972], [-7...
197  4620887.05  [[[-73.43923297693263, 45.571191050160486], [-...
203  4620887.06  [[[-73.4304873632595, 45.569386421452265], [-7...
220  4620887.07  [[[-73.41092211200962, 45.5727511239735], [-73...


MERGE THE DATA TOGETHER

In [23]:
import pandas as pd
# ctuid to latlong
df = pd.read_csv('/content/drive/MyDrive/Ai4Good Project/GentrificAItion/FrontEnd/Final_data/ctuid_to_dings_and_latlong.csv')

print(len(df))
print(df.head())


1006
   Gentrifiable Ding 2021  Gentrifiable Ding 2016  Gentrifiable Ding 2011  \
0                    True                    True                    True   
1                    True                    True                    True   
2                   False                    True                    True   
3                    True                    True                    True   
4                    True                    True                    True   

   Gentrifiable Ding 2006  Gentrifiable Ding 2001  Gentrified Ding 2021  \
0                    True                    True                  True   
1                    True                    True                 False   
2                    True                    True                 False   
3                    True                    True                 False   
4                    True                    True                 False   

   Gentrified Ding 2016  Gentrified Ding 2011  Gentrified Ding 2006  \
0         

In [24]:
# drop latlon_coordinates column
df = df.drop('latlon_coordinates', axis=1)
print(len(df))
print(df.head())

1006
   Gentrifiable Ding 2021  Gentrifiable Ding 2016  Gentrifiable Ding 2011  \
0                    True                    True                    True   
1                    True                    True                    True   
2                   False                    True                    True   
3                    True                    True                    True   
4                    True                    True                    True   

   Gentrifiable Ding 2006  Gentrifiable Ding 2001  Gentrified Ding 2021  \
0                    True                    True                  True   
1                    True                    True                 False   
2                    True                    True                 False   
3                    True                    True                 False   
4                    True                    True                 False   

   Gentrified Ding 2016  Gentrified Ding 2011  Gentrified Ding 2006  \
0         

In [25]:
# add dings from 2026
gentrifiable_2026 = pd.read_csv('/content/drive/MyDrive/Ai4Good Project/GentrificAItion/ModelOutputs/Gentrifiable Ding 2026.csv')
gentrified_2026 = pd.read_csv('/content/drive/MyDrive/Ai4Good Project/GentrificAItion/ModelOutputs/Gentrified Ding 2026.csv')
gen_level_2026 = pd.read_csv('/content/drive/MyDrive/Ai4Good Project/GentrificAItion/ModelOutputs/Gentrification level 2026.csv')

dings_2026 = gentrifiable_2026.merge(gentrified_2026, on='ctuid')
dings_2026 = dings_2026.merge(gen_level_2026, on='ctuid')

print(len(dings_2026))
print(dings_2026.head())

1020
       ctuid  Gentrifiable Ding 2026  Gentrified Ding 2026  \
0       -1.0                       0                     0   
1  4620001.0                       0                     0   
2  4620002.0                       1                     0   
3  4620002.0                       1                     0   
4  4620002.0                       1                     0   

   Gentrification level 2026  
0                          0  
1                          0  
2                          0  
3                          0  
4                          0  


In [26]:
# Drop duplicates, keeping only the first occurrence of each 'ctuid'
dings_2026 = dings_2026.drop_duplicates(subset='ctuid', keep='first')

print(len(dings_2026))

1006


In [27]:
# merge 2026
df = df.merge(dings_2026, on='ctuid')

print(len(df))
print(df.head())

1006
   Gentrifiable Ding 2021  Gentrifiable Ding 2016  Gentrifiable Ding 2011  \
0                    True                    True                    True   
1                    True                    True                    True   
2                   False                    True                    True   
3                    True                    True                    True   
4                    True                    True                    True   

   Gentrifiable Ding 2006  Gentrifiable Ding 2001  Gentrified Ding 2021  \
0                    True                    True                  True   
1                    True                    True                 False   
2                    True                    True                 False   
3                    True                    True                 False   
4                    True                    True                 False   

   Gentrified Ding 2016  Gentrified Ding 2011  Gentrified Ding 2006  \
0         

In [30]:
# add lat long coordinates

latlon = pd.read_csv('/content/drive/MyDrive/Ai4Good Project/GentrificAItion/FrontEnd/Final_data/ALL_ctuid_to_latlong.csv')

latlon = latlon.rename(columns={'CTUID': 'ctuid'})

print(len(latlon))
print(latlon.head())

1004
        ctuid                                 latlon_coordinates
0  4620725.07  [[[-73.9360593674554, 45.58568488364327], [-73...
1  4620652.04  [[[-73.80088052135274, 45.53047772747972], [-7...
2  4620887.05  [[[-73.43923297693263, 45.571191050160486], [-...
3  4620887.06  [[[-73.4304873632595, 45.569386421452265], [-7...
4  4620887.07  [[[-73.41092211200962, 45.5727511239735], [-73...


In [31]:
df = df.merge(latlon, on='ctuid')

print(len(df))
print(df.head())

1006
   Gentrifiable Ding 2021  Gentrifiable Ding 2016  Gentrifiable Ding 2011  \
0                    True                    True                    True   
1                    True                    True                    True   
2                   False                    True                    True   
3                    True                    True                    True   
4                    True                    True                    True   

   Gentrifiable Ding 2006  Gentrifiable Ding 2001  Gentrified Ding 2021  \
0                    True                    True                  True   
1                    True                    True                 False   
2                    True                    True                 False   
3                    True                    True                 False   
4                    True                    True                 False   

   Gentrified Ding 2016  Gentrified Ding 2011  Gentrified Ding 2006  \
0         

In [32]:
df.to_csv('/content/drive/MyDrive/Ai4Good Project/GentrificAItion/FrontEnd/Final_data/FINAL DATA FOR WEBSITE/ctuid_to_dings_and_coordinates.csv', index=False)

In [35]:
# make geojson

# Load the CSV file
file_path = '/content/drive/MyDrive/Ai4Good Project/GentrificAItion/FrontEnd/Final_data/FINAL DATA FOR WEBSITE/ctuid_to_dings_and_coordinates.csv'
df = pd.read_csv(file_path)

In [33]:
import json

def create_geojson(df):
    geojson = {
        "type": "FeatureCollection",
        "features": []
    }

    years = [2001, 2006, 2011, 2016, 2021, 2026]
    gentrifiable_columns = {
        2001: 'Gentrifiable Ding 2001',
        2006: 'Gentrifiable Ding 2006',
        2011: 'Gentrifiable Ding 2011',
        2016: 'Gentrifiable Ding 2016',
        2021: 'Gentrifiable Ding 2021',
        2026: 'Gentrifiable Ding 2026'
    }
    gentrified_columns = {
        2001: None,
        2006: 'Gentrified Ding 2006',
        2011: 'Gentrified Ding 2011',
        2016: 'Gentrified Ding 2016',
        2021: 'Gentrified Ding 2021',
        2026: 'Gentrified Ding 2026'
    }
    gentrification_level_columns = {
        2001: None,
        2006: 'Gentrification Level Ding 2006',
        2011: 'Gentrification Level Ding 2011',
        2016: 'Gentrification Level Ding 2016',
        2021: 'Gentrification Level Ding 2021',
        2026: 'Gentrification level 2026'
    }

    for index, row in df.iterrows():
        tract_id = row['ctuid']
        coordinates = json.loads(row['latlon_coordinates'].replace("'", "\""))

        for year in years:
            feature = {
                "type": "Feature",
                "geometry": {
                    "type": "Polygon",
                    "coordinates": coordinates
                },
                "properties": {
                    "tract_id": tract_id,
                    "year": year,
                    "gentrified": row[gentrified_columns[year]] if gentrified_columns[year] else None,
                    "gentrifiable": row[gentrifiable_columns[year]],
                    "gentrification_level": row[gentrification_level_columns[year]] if gentrification_level_columns[year] else None
                }
            }
            geojson["features"].append(feature)

    return geojson



In [34]:
# Create the GeoJSON structure
geojson = create_geojson(df)


In [36]:
# Save the GeoJSON to a file
output_file_path = '/content/drive/MyDrive/Ai4Good Project/GentrificAItion/FrontEnd/Final_data/FINAL DATA FOR WEBSITE/data.geojson'
with open(output_file_path, 'w') as geojson_file:
    json.dump(geojson, geojson_file, indent=4)
